# Practice: Budget a Project in Tokens

**Computational Text Analysis Pack · Module 25 — Transformers and Attention**

*Technique:* Costing a real corpus across languages and models before committing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/12-neural-foundations/25c_practice_tokeniser_budget.ipynb)

Data: `masakhanews.csv`, `colonial_texts.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv', 'colonial_texts.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
df_colonial = pd.read_csv('colonial_texts.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## The question

> **You want to run a transformer over your whole corpus. What will it cost, how
> long will it take, and will it cost the same for every language in it?**

Module 25b measured fertility. This turns that into a project plan, which is the
form the information is actually useful in.

### How to use this notebook

A worked case, not a tutorial. Run it once for the shape of the argument, then
change the question at the top to one of your own.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. It does not
    match combining marks, and a tone mark on ọ or ẹ is always one, so the
    pattern adds the combining-accent range U+0300-U+036F; without it ọ̀rọ̀
    splits into ọ + rọ. The common shortcut `[a-z]+` throws away far more and
    quietly changes every count in this notebook — see notebook 03b.
    """
    return re.findall(r"[\w\u0300-\u036f]+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé lórí ọ̀rọ̀ ẹ̀kọ́ — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("bare \\w+   :", re.findall(r"\w+", sample.lower()))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter; bare \\w+")
print("splits them wherever a tone mark sits on ọ or ẹ.")

In [ ]:
from transformers import AutoTokenizer
import numpy as np

MODELS = {"xlm-roberta": "FacebookAI/xlm-roberta-base",
          "afro-xlmr": "Davlan/afro-xlmr-base"}
toks = {}
for label, repo in MODELS.items():
    try:
        toks[label] = AutoTokenizer.from_pretrained(repo)
        print(f"loaded {label}")
    except Exception as e:
        print(f"skip {label}: {str(e)[:60]}")

In [ ]:
# Step 1: measure fertility on YOUR corpus, not on a published table.
rows = []
for lang, g in df.groupby("language"):
    text = " ".join(g["text"].head(30).astype(str))[:15000]
    w = len(tokenise(text))
    if w < 400: continue
    r = {"language": lang, "words": w,
         "corpus_words": int(df[df["language"] == lang]["n_words"].sum())}
    for label, tk in toks.items():
        r[label] = len(tk.tokenize(text)) / w
    rows.append(r)
fert = pd.DataFrame(rows)
main = list(toks)[0] if toks else None
if main:
    fert = fert.sort_values(main, ascending=False)
    print(fert[["language", "corpus_words", main] + ([list(toks)[1]] if len(toks) > 1 else [])]
          .to_string(index=False, float_format=lambda v: f"{v:.2f}"))

In [ ]:
# Step 2: turn fertility into tokens, time and money.
GBP_PER_MILLION = 0.15      # <- your provider's rate
TOKENS_PER_SEC = 2000       # <- measure this on your hardware

if main:
    fert["est_tokens"] = fert["corpus_words"] * fert[main]
    fert["gbp"] = fert["est_tokens"] / 1e6 * GBP_PER_MILLION
    fert["minutes"] = fert["est_tokens"] / TOKENS_PER_SEC / 60
    out = fert[["language", "corpus_words", main, "est_tokens", "gbp", "minutes"]]
    print(out.to_string(index=False, float_format=lambda v: f"{v:,.2f}"))
    print()
    print(f"WHOLE CORPUS: {fert['est_tokens'].sum():,.0f} tokens, "
          f"GBP {fert['gbp'].sum():,.2f}, {fert['minutes'].sum():,.0f} minutes")
    print()
    hi, lo = fert.iloc[0], fert.iloc[-1]
    print(f"Per word, {hi['language']} costs {hi[main]/lo[main]:.1f}x what "
          f"{lo['language']} costs. Same corpus, same model, same fee schedule.")

In [ ]:
if main:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    axes[0].barh(fert["language"][::-1], fert[main][::-1], color="steelblue")
    axes[0].axvline(1.0, ls="--", color="grey", lw=1)
    axes[0].set_xlabel("tokens per word"); axes[0].set_title("Fertility")
    axes[1].barh(fert["language"][::-1], fert["gbp"][::-1], color="darkorange")
    axes[1].set_xlabel("estimated GBP"); axes[1].set_title("Cost for this corpus")
    for a in axes: a.tick_params(labelsize=7)
    plt.tight_layout()

## What the budget tells you beyond the number

**The cost is not evenly distributed, and it falls hardest on the languages with
least digital text.** A project that processes everything spends
disproportionately on the languages that are already worst served — and a
project that trims the budget by dropping the expensive languages makes the
representation problem worse.

That is a research-design decision with an ethical dimension, and it is visible
only if you cost per language rather than in aggregate.

**Defensible:** "Processing the corpus required an estimated [N] tokens at a
mean fertility of [X] (range [lo]–[hi] across languages); costs and coverage per
language are reported in Table [n]."

**Not defensible:** "we ran the model over the corpus", with a total cost and no
per-language breakdown.

### 🔧 Your turn 1

Recompute the budget with `output_sequence_length` truncation at 256 tokens. How much text do you lose per language?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

Truncation is a hidden, uneven data-loss step: at a fixed token limit, a
high-fertility language loses far more *words* than a low-fertility one, so your
effective sample differs by language in a way no methods section usually
mentions. If you truncate — and almost everyone does — report the words retained
per language, not just the token limit. It is the same class of silent bias as
module 3's ASCII trap, arriving through the budget rather than the tokeniser.

</details>